<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="./images/btp-banner.gif" alt="BTP A&C">
</div>

## Retrieval-Augmented Generation with SAP HANA Cloud Vector Engine

In this demo, we will explore how to enhance the capabilities of Large Language Models (LLMs) with **SAP HANA Cloud vector engine**. You will learn how to embed unstructured and semi-structured data using LLMs from **SAP Generative AI Hub**, store the vector embeddings in **SAP HANA Cloud**. Additionally, you will query vector embeddings, and forward the relevant results to a LLM for an augmented answer.

This improved version includes **production-ready best practices** such as comprehensive error handling, centralized configuration, proper logging, resource management, and input validation.

## 🎯 Learning Objectives
By the end of this demo, you will be able to:
- Implement a full RAG pipeline using Python, LangChain, and Generative AI Hub SDK with production-ready best practices
- Generate embeddings for document chunks using Generative AI Hub SDK with proper error handling
- Retrieve the most relevant content based on semantic similarity using SAP HANA Cloud similarity search
- Augment user prompts with retrieved context and invoke LLMs (e.g., GPT-5) to generate more accurate, grounded answers
- Design and use prompt templates to enhance the quality of generated responses
- Implement proper resource management, logging, and configuration management

## 🚨 Requirements

Please review the following requirements before starting the demo:
- Enable the additional feature **Natural Language Processing (NLP)** in your SAP HANA Cloud database
- Deploy Large Language Models (LLMs): **text-embedding-3-large** and **gpt-5** in SAP AI Launchpad
- Configure environment variables in `.env` file with required credentials

## 📝 About the Data

The data set is a product catalog of IT accessory products. Here are the main attributes and their descriptions based on the sample data:

| Field | Description |
|-------|-------------|
| **PRODUCT_ID** | A unique identifier for each product |
| **PRODUCT_NAME** | The name of the product, which typically includes the brand and the model |
| **CATEGORY** | The general category of the product, which is "IT Accessories" for all entries sampled |
| **DESCRIPTION** | A detailed description of the product, highlighting key features and specifications |
| **UNIT_PRICE** | The price of the product in Euros |
| **UNIT_MEASURE** | The unit of measure for the product, typically "Each" indicating pricing per item |
| **SUPPLIER_ID** | A unique identifier for the supplier of the product |
| **SUPPLIER_NAME** | The name of the supplier |
| **LEAD_TIME_DAYS** | The number of days it takes from order to delivery |
| **MIN_ORDER** | The minimum order quantity required |
| **CURRENCY** | The currency of the transaction, which is "EURO" for all entries |
| **SUPPLIER_COUNTRY** | The country where the supplier is located, which is "Germany" for all sampled entries |
| **SUPPLIER_ADDRESS** | The physical address of the supplier |
| **AVAILABILITY_DAYS** | The number of days the product is available for delivery |
| **SUPPLIER_CITY** | The city where the supplier is located |
| **STOCK_QUANTITY** | The quantity of the product currently in stock |
| **MANUFACTURER** | The company that manufactured the product |
| **CITY_LAT** | Geographical coordinates of the city (latitude) |
| **CITY_LONG** | Geographical coordinates of the city (longitude) |
| **RATING** | A rating for the product, on a scale from 1 to 5 |

This dataset is structured to support various business operations such as inventory management, order processing, and logistics planning, providing a comprehensive view of product offerings, supplier details, and stock levels.

### Step 1: Install Python packages

Run the following package installations. **pip** is the package installer for Python. You can use pip to install packages from the Python Package Index and other indexes.

⚠️ **Note:** Jupyter Notebook kernel restart required after package installation.

In [ ]:
%pip install -U hdbcli
%pip install -U sap-ai-sdk-gen[all]
%pip install -U langchain-hana
%pip install -U langchain-core
%pip install -U langchain-community
%pip install -U langchain-text-splitters
%pip install -U folium
%pip install -U ipywidgets
%pip install -U python-dotenv

# kernel restart required!!!

### Step 2: Import required libraries and setup logging

Import all necessary modules and configure proper logging for the application.

**What we're importing:**
- `os`, `json`: For environment variables and JSON handling
- `logging`, `warnings`: For structured logging and warning suppression
- `typing`: For type hints to improve code clarity
- `dotenv`: To load environment variables from .env file

**Why logging matters:** Using the logging module instead of print statements provides better control over output levels, log formatting, and makes debugging easier in production environments.

In [ ]:
import os
import json
import logging
import warnings
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv

# Suppress deprecation warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Load environment variables
load_dotenv()

logger.info("✅ Libraries imported and logging configured successfully!")
print("✅ Libraries imported and logging configured successfully!")

### Step 3: Configure centralized settings

Establish a centralized configuration dictionary that controls all aspects of the RAG system. This approach enables easy experimentation with different parameters without modifying code throughout the notebook.

In [ ]:
# Centralized configuration
CONFIG = {
    # Model Configuration
    "embedding_model": "text-embedding-3-large",
    "llm_model": "gpt-5",
    "temperature": 0.7,
    "max_retries": 10,
    
    # Document Processing
    "chunk_size": 1000,
    "chunk_overlap": 200,
    "embedding_chunk_size": 100,
    
    # Retrieval Configuration
    "top_k_results": 20,
    
    # Database Configuration
    "hana_port": 443,
    "ssl_validate": False,
    "autocommit": True,
    
    # File Paths
    "csv_file": "data/new_product.csv",
    "csv_delimiter": ";",
}

logger.info("Configuration loaded successfully")
print("✅ Configuration loaded successfully!")
print(json.dumps(CONFIG, indent=2))

### Step 4: Validate environment variables

Before proceeding, validate that all required environment variables are properly configured. This prevents cryptic errors later in the pipeline.

In [ ]:
def validate_environment() -> None:
    """
    Validate that all required environment variables are set.
    
    Raises:
        ValueError: If any required environment variable is missing
    """
    required_vars = [
        'AICORE_AUTH_URL',
        'AICORE_CLIENT_ID',
        'AICORE_CLIENT_SECRET',
        'AICORE_RESOURCE_GROUP',
        'AICORE_BASE_URL',
        'HANA_VECTOR_USER',
        'HANA_VECTOR_PASS',
        'HANA_VECTOR_HOST'
    ]
    
    missing_vars = [var for var in required_vars if not os.getenv(var)]
    
    if missing_vars:
        error_msg = f"Missing required environment variables: {', '.join(missing_vars)}"
        logger.error(error_msg)
        raise ValueError(error_msg)
    
    logger.info("All required environment variables are set")
    print("✅ All environment variables validated successfully!")

# Validate environment
try:
    validate_environment()
except ValueError as e:
    print(f"❌ Environment validation failed: {str(e)}")
    print("Please ensure all required variables are set in your .env file")
    raise

### Step 5: Configure AI Core Client with error handling

Execute the configuration module below to enable access to SAP's Generative AI foundation models. This implementation includes comprehensive error handling to provide clear feedback if connection fails.

In [ ]:
from ai_core_sdk.ai_core_v2_client import AICoreV2Client
from gen_ai_hub.proxy.gen_ai_hub_proxy import GenAIHubProxyClient

try:
    # Get credentials from environment
    URL = os.getenv('AICORE_AUTH_URL')
    CLIENT_ID = os.getenv('AICORE_CLIENT_ID')
    CLIENT_SECRET = os.getenv('AICORE_CLIENT_SECRET')
    RESOURCE_GROUP = os.getenv('AICORE_RESOURCE_GROUP')
    API_URL = os.getenv('AICORE_BASE_URL')

    # Set up the AICoreV2Client
    ai_core_client = AICoreV2Client(
        base_url=API_URL,
        auth_url=URL,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
        resource_group=RESOURCE_GROUP
    )

    # Initialize GenAIHub proxy client
    proxy_client = GenAIHubProxyClient(ai_core_client=ai_core_client)
    
    logger.info("AI Core client initialized successfully")
    print("✅ AI Core Client connection is established successfully!")
    
except Exception as e:
    logger.error(f"Failed to initialize AI Core client: {str(e)}")
    print(f"❌ Failed to connect to AI Core: {str(e)}")
    raise

Test the AI Core configuration by making a call to the **text-embedding-3-large** model

In [ ]:
from gen_ai_hub.proxy.native.openai import embeddings

try:
    response = embeddings.create(
        input="SAP Generative AI Hub is awesome!",
        model_name=CONFIG["embedding_model"]
    )
    
    logger.info(f"Test embedding generated successfully. Dimension: {len(response.data[0].embedding)}")
    print(f"✅ Test embedding successful! Vector dimension: {len(response.data[0].embedding)}")
    
except Exception as e:
    logger.error(f"Failed to generate test embedding: {str(e)}")
    print(f"❌ Embedding test failed: {str(e)}")
    raise

Initialize the embeddings model. This model will be used to generate embeddings for both documents and user prompts.

In [ ]:
from gen_ai_hub.proxy.langchain import OpenAIEmbeddings

try:
    embedding_model = OpenAIEmbeddings(
        proxy_model_name=CONFIG["embedding_model"],
        chunk_size=CONFIG["embedding_chunk_size"],
        max_retries=CONFIG["max_retries"]
    )
    
    logger.info(f"Embedding model '{CONFIG['embedding_model']}' initialized")
    print("✅ Embedding model is initialized successfully!")
    
except Exception as e:
    logger.error(f"Failed to initialize embedding model: {str(e)}")
    print(f"❌ Failed to initialize embedding model: {str(e)}")
    raise

Initialize the LLM model. This model is used for generating responses or interacting in a chat-like environment.

In [ ]:
from gen_ai_hub.proxy.langchain import ChatOpenAI

try:
    llm = ChatOpenAI(
        proxy_model_name=CONFIG["llm_model"],
        proxy_client=proxy_client,
        temperature=CONFIG["temperature"]
    )
    
    logger.info(f"LLM model '{CONFIG['llm_model']}' initialized")
    print("✅ LLM model is initialized successfully!")
    
except Exception as e:
    logger.error(f"Failed to initialize LLM model: {str(e)}")
    print(f"❌ Failed to initialize LLM model: {str(e)}")
    raise

### Step 6: Ask LLM without business context

After completing the configuration we are ready to ask the first question directly to LLM without any business product context. The response will be arbitrary and not relate to our product data.

> **Note**: This output allows us to compare how RAG-enhanced responses differ from direct LLM responses.

In [ ]:
from IPython.display import Markdown, display

question = "Suggest a keyboard with a rating 4 or more"

try:
    response = llm.invoke(question)
    logger.info("LLM response generated without RAG context")
    print("\n📝 Response without RAG context:\n")
    display(Markdown(response.content))
    
except Exception as e:
    logger.error(f"Failed to get LLM response: {str(e)}")
    print(f"❌ Failed to get response: {str(e)}")

### Step 7: Prepare and process documentation

Load and process text data from CSV and PDF files. This implementation includes error handling and validation of file existence before processing.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders.csv_loader import CSVLoader
from pathlib import Path

def load_and_split_csv(file_path: str) -> List[Any]:
    """
    Load and split CSV file into document chunks.
    
    Args:
        file_path: Path to the CSV file
        
    Returns:
        List of document chunks
        
    Raises:
        FileNotFoundError: If the CSV file doesn't exist
        Exception: If processing fails
    """
    if not Path(file_path).exists():
        raise FileNotFoundError(f"CSV file not found: {file_path}")
    
    try:
        loader = CSVLoader(
            file_path=file_path,
            encoding="utf-8",
            csv_args={
                "delimiter": CONFIG["csv_delimiter"],
                "quotechar": '"'
            },
        )
        
        documents = loader.load()
        text_splitter = CharacterTextSplitter(
            chunk_size=CONFIG["chunk_size"],
            chunk_overlap=0
        )
        chunks = text_splitter.split_documents(documents)
        
        logger.info(f"CSV file loaded: {len(chunks)} chunks created")
        return chunks
        
    except Exception as e:
        logger.error(f"Failed to process CSV file: {str(e)}")
        raise

# Process CSV data
try:
    csv_chunks = load_and_split_csv(CONFIG["csv_file"])
    print(f"✅ CSV processed: {len(csv_chunks)} document chunks created")
    
    # Show sample
    if csv_chunks:
        print(f"\nSample chunk:")
        print(f"Metadata: {csv_chunks[0].metadata}")
        print(f"Content preview: {csv_chunks[0].page_content[:200]}...")
        
except Exception as e:
    print(f"❌ Failed to process CSV: {str(e)}")
    csv_chunks = []

### Step 8: Connect to SAP HANA Cloud Database

Establish a secure connection to SAP HANA Cloud with proper error handling and connection validation.

In [ ]:
from hdbcli import dbapi

try:
    # Get HANA credentials
    HANA_USER = os.getenv('HANA_VECTOR_USER')
    HANA_PASS = os.getenv('HANA_VECTOR_PASS')
    HANA_HOST = os.getenv('HANA_VECTOR_HOST')

    # Establish connection
    connection = dbapi.connect(
        address=HANA_HOST,
        port=CONFIG["hana_port"],
        user=HANA_USER,
        password=HANA_PASS,
        autocommit=CONFIG["autocommit"],
        sslValidateCertificate=CONFIG["ssl_validate"],
    )
    
    # Test connection
    cursor = connection.cursor()
    cursor.execute("SELECT 1 FROM DUMMY")
    cursor.fetchone()
    cursor.close()
    
    logger.info("HANA Cloud connection established and validated")
    print("✅ HANA Cloud connection is established successfully!")
    
except Exception as e:
    logger.error(f"Failed to connect to HANA Cloud: {str(e)}")
    print(f"❌ HANA Cloud connection failed: {str(e)}")
    raise

### Step 9: Create vector store with error handling

Create a LangChain VectorStore interface for the HANA database. This implementation includes proper table management and error handling.

In [ ]:
from langchain_hana import HanaDB

def create_vector_store(table_name: str) -> HanaDB:
    """
    Create a vector store in HANA Cloud.
    
    Args:
        table_name: Name of the table to create
        
    Returns:
        HanaDB: Initialized vector store
        
    Raises:
        Exception: If vector store creation fails
    """
    try:
        vector_store = HanaDB(
            embedding=embedding_model,
            connection=connection,
            table_name=table_name,
            content_column="VEC_TEXT",
            metadata_column="VEC_META",
            vector_column="VEC_VECTOR"
        )
        
        logger.info(f"Vector store created: {table_name}")
        return vector_store
        
    except Exception as e:
        logger.error(f"Failed to create vector store: {str(e)}")
        raise

# Create vector store
try:
    table_name = f"PRODUCT_IT_ACCESSORY_{HANA_USER}"
    vector_table = create_vector_store(table_name)
    print(f"✅ Vector Database connection established: {table_name}")
    
except Exception as e:
    print(f"❌ Failed to create vector store: {str(e)}")
    raise

Populate the vector store with document chunks

In [ ]:
try:
    # Combine all chunks
    all_chunks = csv_chunks
    
    if not all_chunks:
        raise ValueError("No document chunks available to add to vector store")
    
    # Delete existing documents
    vector_table.delete(filter={})
    logger.info("Existing documents deleted from vector store")
    
    # Add document chunks
    vector_table.add_documents(all_chunks)
    logger.info(f"Added {len(all_chunks)} documents to vector store")
    
    print(f"✅ Vector store populated with {len(all_chunks)} document chunks!")
    
except Exception as e:
    logger.error(f"Failed to populate vector store: {str(e)}")
    print(f"❌ Failed to populate vector store: {str(e)}")
    raise

Verify the embeddings in the vector store

In [ ]:
try:
    cursor = connection.cursor()
    sql = f'SELECT COUNT(*) FROM "{vector_table.table_name}"'
    cursor.execute(sql)
    count = cursor.fetchone()[0]
    cursor.close()
    
    logger.info(f"Verified {count} vectors in store")
    print(f"✅ Verified: {count} vectors stored in database")
    
except Exception as e:
    logger.error(f"Failed to verify vector store: {str(e)}")
    print(f"⚠️ Could not verify vector count: {str(e)}")

### Step 10: Define prompt template

Define a prompt template to provide context for generating accurate, structured responses. The template ensures consistent output formatting.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Define the prompt template
prompt_template = """Use the following pieces of context to answer the question at the end. If you don't know the answer,
just say you don't know, don't try to make up an answer.

Format the results in a list of JSON items with the following keys:
"PRODUCT_ID", "PRODUCT_NAME", "CATEGORY", "DESCRIPTION", "UNIT_PRICE", "UNIT_MEASURE", 
"SUPPLIER_ID", "SUPPLIER_NAME", "LEAD_TIME_DAYS", "MIN_ORDER", "CURRENCY", 
"SUPPLIER_COUNTRY", "SUPPLIER_ADDRESS", "SUPPLIER_CITY", "CITY_LAT", "CITY_LONG", "RATING"

The 'RATING' key value is an integer datatype ranging from 0 stars to 5 stars, where 0 stars is 'bad' and 5 stars is 'excellent'.
Do not include json markdown codeblock syntax in the results (no ```json ``` markers).

Context:
{context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(prompt_template)

logger.info("Prompt template configured")
print("✅ Prompt template configured successfully!")

### Step 11: Generate RAG-enhanced answers

Create a retrieval-based QA system that combines vector search with LLM generation for accurate, context-aware responses.

**What is LCEL (LangChain Expression Language)?**

LCEL is a declarative way to compose LangChain components into chains. It uses the pipe operator (`|`) to chain components together, similar to Unix pipes. Benefits include:

- **Readability**: Clear, linear flow of data through components
- **Streaming Support**: Built-in support for streaming responses
- **Async Support**: Native async/await compatibility
- **Parallel Execution**: Automatic parallelization where possible
- **Type Safety**: Better type hints and IDE support

**Our RAG Chain Flow:**
1. Question → Retriever (fetch relevant documents)
2. Documents → format_docs (combine into context string)
3. Context + Question → Prompt Template (format prompt)
4. Formatted Prompt → LLM (generate response)
5. LLM Response → String Parser (extract text)

In [ ]:
def format_docs(docs):
    """
    Format retrieved documents into a single string for context.
    
    Args:
        docs: List of retrieved documents
        
    Returns:
        Formatted string containing all document contents
    """
    return "\n\n".join(doc.page_content for doc in docs)

try:
    # Create retriever from vector store
    retriever = vector_table.as_retriever(
        search_kwargs={'k': CONFIG["top_k_results"]}
    )
    
    # Create RAG chain using LCEL
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    
    logger.info(f"RAG chain created with top_k={CONFIG['top_k_results']}")
    print("✅ RAG chain created successfully using LCEL!")
    
except Exception as e:
    logger.error(f"Failed to create RAG chain: {str(e)}")
    print(f"❌ Failed to create RAG chain: {str(e)}")
    raise

Execute a query with the RAG system

In [ ]:
question = "Find products with a rating of 4 and more."

try:
    logger.info(f"Executing RAG query: {question}")
    
    # Invoke the RAG chain
    answer = rag_chain.invoke(question)
    
    # Get source documents for reference
    source_docs = retriever.invoke(question)
    
    logger.info(f"RAG query successful. Retrieved {len(source_docs)} source documents")
    
    print("\n" + "="*80)
    print("📊 RAG-Enhanced Answer:")
    print("="*80)
    print(answer)
    print("\n" + "="*80)
    print(f"📚 Used {len(source_docs)} source documents")
    print("="*80)
    
except Exception as e:
    logger.error(f"RAG query failed: {str(e)}")
    print(f"❌ Query failed: {str(e)}")
    raise

## 🚧 Optional Exercise: SAP HANA Cloud Multi-modeling with Spatial and Vector Engines

In this section, we explore combining vector embeddings with spatial data within the same SQL environment. You will learn how to define tables including both vector embeddings and spatial data in SAP HANA Cloud, and visualize warehouse locations on a map.

### Optional Step 1: Save RAG results to HANA Document Store

Store RAG results in a JSON document collection for further processing and analysis.

In [ ]:
import json

def create_document_collection(collection_name: str, documents: List[Dict]) -> None:
    """
    Create and populate a document collection in HANA.
    
    Args:
        collection_name: Name of the collection to create
        documents: List of document dictionaries to insert
    """
    try:
        cursor = connection.cursor()
        
        # Drop and recreate collection
        try:
            cursor.execute(f"DROP COLLECTION {collection_name}")
            logger.info(f"Dropped existing collection: {collection_name}")
        except:
            pass
        
        cursor.execute(f"CREATE COLLECTION {collection_name}")
        logger.info(f"Created collection: {collection_name}")
        
        # Insert documents
        for doc in documents:
            json_str = json.dumps(doc)
            cursor.execute(f"INSERT INTO {collection_name} VALUES ('{json_str}')")
        
        logger.info(f"Inserted {len(documents)} documents into {collection_name}")
        cursor.close()
        print(f"✅ Created and populated collection: {collection_name}")
        
    except Exception as e:
        logger.error(f"Failed to create document collection: {str(e)}")
        raise

# Parse and store RAG results
try:
    products = json.loads(answer)
    collection_name = "GX_RAG_PRODUCTS_DEV"
    create_document_collection(collection_name, products)
    
except Exception as e:
    print(f"⚠️ Could not create document collection: {str(e)}")

### Optional Step 2: Create table with spatial data

Create a table that combines product information with geospatial location data.

In [ ]:
try:
    cursor = connection.cursor()
    table_name = "GX_SUPPLIER_LOCATIONS_DEV"
    
    # Drop existing table
    try:
        cursor.execute(f"DROP TABLE {table_name}")
    except:
        pass
    
    # Create table with spatial column
    sql = f"""CREATE TABLE {table_name} AS (
        SELECT SUPPLIER_ID,
               SUPPLIER_CITY,
               NEW ST_POINT(TO_DOUBLE(CITY_LONG), TO_DOUBLE(CITY_LAT)) AS SUPPLIER_LOCATION,
               SUPPLIER_ADDRESS,
               PRODUCT_NAME,
               RATING
        FROM {collection_name}
    )"""
    
    cursor.execute(sql)
    cursor.close()
    
    logger.info(f"Created spatial table: {table_name}")
    print(f"✅ Created spatial table: {table_name}")
    
except Exception as e:
    logger.error(f"Failed to create spatial table: {str(e)}")
    print(f"⚠️ Could not create spatial table: {str(e)}")

### Optional Step 3: Query and visualize spatial data

Retrieve geospatial data and visualize supplier locations on an interactive map.

In [ ]:
try:
    cursor = connection.cursor()
    table_name = "GX_SUPPLIER_LOCATIONS_DEV"
    
    sql = f"""SELECT SUPPLIER_ID, SUPPLIER_CITY,
                     SUPPLIER_LOCATION.ST_Y() as latitudes,
                     SUPPLIER_LOCATION.ST_X() as longitudes,
                     SUPPLIER_ADDRESS, PRODUCT_NAME, RATING
              FROM {table_name}"""
    
    cursor.execute(sql)
    points = cursor.fetchall()
    cursor.close()
    
    logger.info(f"Retrieved {len(points)} spatial data points")
    print(f"✅ Retrieved {len(points)} supplier locations")
    
except Exception as e:
    logger.error(f"Failed to query spatial data: {str(e)}")
    print(f"⚠️ Could not query spatial data: {str(e)}")
    points = []

In [ ]:
import folium

if points:
    try:
        # Calculate average location for map center
        average_lat = sum([item[2] for item in points]) / len(points)
        average_lon = sum([item[3] for item in points]) / len(points)
        
        # Create map
        map_obj = folium.Map(location=[average_lat, average_lon], zoom_start=7)
        
        # Add markers
        for supplier_id, city, lat, lon, address, product, rating in points:
            popup_text = f"""Supplier ID: {supplier_id}<br>
                            City: {city}<br>
                            Address: {address}<br>
                            Product: {product}<br>
                            Rating: {rating}"""
            
            folium.Marker(
                [lat, lon],
                popup=folium.Popup(popup_text, max_width=300, min_width=300)
            ).add_to(map_obj)
        
        logger.info("Map visualization created successfully")
        print("✅ Map created with supplier locations")
        display(map_obj)
        
    except Exception as e:
        logger.error(f"Failed to create map: {str(e)}")
        print(f"⚠️ Could not create map: {str(e)}")
else:
    print("⚠️ No spatial data available for visualization")

### Final Cleanup

Close all database connections and clean up resources properly. This ensures no lingering connections remain and resources are freed.

In [ ]:
def cleanup_resources() -> None:
    """
    Clean up database connections and resources.
    
    This function safely closes the HANA database connection
    and performs any necessary cleanup operations.
    """
    try:
        if 'connection' in globals() and connection:
            connection.close()
            logger.info("HANA Cloud connection closed")
            print("✅ Database connection closed successfully")
    except Exception as e:
        logger.error(f"Error during cleanup: {str(e)}")
        print(f"⚠️ Cleanup warning: {str(e)}")

# Execute cleanup
cleanup_resources()
print("\n🎉 Demo completed successfully with best practices applied!")